# Aula 22 — Redução de dimensionalidade em ML

Laboratório reproduzível para separar três usos diferentes: PCA como transformação linear, t-SNE como mapa local e o grafo fuzzy que antecede o layout do UMAP.

**Protocolo:** dados locais ou sintéticos, seed fixa, teste reservado quando há alvo e nenhuma credencial.

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/03-machine-learning/notebooks/22-reducao-dimensionalidade-ml-laboratorio.ipynb)

## Dependências

- Python ≥ 3.11
- NumPy ≥ 2.0
- SciPy ≥ 1.13
- scikit-learn ≥ 1.5
- Matplotlib ≥ 3.8

O experimento implementa o grafo fuzzy do UMAP sem exigir `umap-learn`. Isso permite auditar a construção das arestas; não equivale ao layout 2D completo do UMAP.

In [ ]:
import platform
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import scipy
import sklearn
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from scipy.stats import spearmanr
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE, trustworthiness
from sklearn.metrics import accuracy_score, pairwise_distances
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("error")
SEED = 20260908
rng = np.random.default_rng(SEED)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seed:", SEED)

## 1. PCA por SVD

Para uma matriz centrada $X_c\in\mathbb{R}^{n\times p}$, a decomposição $X_c=U\Sigma V^\top$ fornece as direções principais nas linhas de $V^\top$. A variância explicada pela componente $j$ é $\sigma_j^2/(n-1)$.

In [ ]:
n = 900
z = rng.normal(size=(n, 3))
mix = np.array([
    [1.0, 0.4, 0.0],
    [0.9, 0.3, 0.1],
    [0.1, 1.2, 0.2],
    [0.0, 0.2, 0.5],
])
X = z @ mix.T + rng.normal(scale=0.08, size=(n, 4))
Xc = X - X.mean(axis=0)

U, singular_values, Vt = np.linalg.svd(Xc, full_matrices=False)
eigenvalues = singular_values**2 / (n - 1)
ratio_manual = eigenvalues / eigenvalues.sum()

pca = PCA().fit(X)
np.testing.assert_allclose(ratio_manual, pca.explained_variance_ratio_, atol=1e-12)
np.testing.assert_allclose(np.abs(Vt), np.abs(pca.components_), atol=1e-12)
print("Variância explicada:", np.round(ratio_manual, 6))
print("Acumulada nas 2 primeiras:", round(float(ratio_manual[:2].sum()), 6))

### Reconstrução

Com $k$ componentes, projetamos por $Z_k=X_cV_k^\top$ e reconstruímos por $\hat X=Z_kV_k+\bar x$. A reconstrução melhora monotonicamente quando $k$ cresce, mas isso não garante melhor previsão do alvo.

In [ ]:
errors = []
for k in range(1, X.shape[1] + 1):
    scores = Xc @ Vt[:k].T
    reconstructed = scores @ Vt[:k] + X.mean(axis=0)
    errors.append(np.mean((X - reconstructed) ** 2))

assert np.all(np.diff(errors) <= 1e-14)
assert errors[-1] < 1e-25
print("MSE de reconstrução por k:", np.round(errors, 8))

## 2. Unidades alteram o PCA

PCA centra os dados, mas não padroniza as features. Multiplicar uma coluna por 100 muda a matriz de covariância e pode fazer a primeira componente apontar quase toda para essa coluna.

In [ ]:
X_units = X.copy()
X_units[:, 3] *= 100

raw_pc1 = PCA(n_components=1).fit(X_units).components_[0]
scaled_model = make_pipeline(StandardScaler(), PCA(n_components=1)).fit(X_units)
scaled_pc1 = scaled_model[-1].components_[0]

raw_dominance = float(np.max(np.abs(raw_pc1)))
scaled_dominance = float(np.max(np.abs(scaled_pc1)))
assert raw_dominance > 0.99
assert scaled_dominance < 0.85
print("Maior |loading| sem escala:", round(raw_dominance, 6))
print("Maior |loading| com escala:", round(scaled_dominance, 6))

## 3. Variância não é relevância para o alvo

O exemplo a seguir contém uma feature de baixa variância que separa as classes e duas features de alta variância que são ruído. O teste fica reservado desde o início.

In [ ]:
n = 2400
y = rng.integers(0, 2, size=n)
signal = (2 * y - 1) * 0.45 + rng.normal(0, 0.12, size=n)
X_target = np.column_stack([
    rng.normal(0, 20, size=n),
    rng.normal(0, 8, size=n),
    signal,
])
X_train, X_test, y_train, y_test = train_test_split(
    X_target, y, test_size=0.30, stratify=y, random_state=SEED
)

full_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED))
pca_one = make_pipeline(PCA(n_components=1), LogisticRegression(max_iter=1000, random_state=SEED))
full_model.fit(X_train, y_train)
pca_one.fit(X_train, y_train)

acc_full = accuracy_score(y_test, full_model.predict(X_test))
acc_pca1 = accuracy_score(y_test, pca_one.predict(X_test))
variance_pca1 = float(pca_one[0].explained_variance_ratio_[0])

assert acc_full > 0.99
assert variance_pca1 > 0.85
assert acc_pca1 < 0.58
print(f"Modelo com todas as features: accuracy={acc_full:.6f}")
print(f"PCA(1): variância={variance_pca1:.6f}; accuracy={acc_pca1:.6f}")

## 4. O PCA aprende estado: ajuste somente no treino

Média, escala e componentes são parâmetros aprendidos. Uma mudança de distribuição no teste deixa explícita a diferença entre o protocolo correto e o ajuste contaminado.

In [ ]:
X_tr = rng.normal(0, 1, size=(600, 5))
X_te = rng.normal(0, 1, size=(300, 5))
X_te[:, 0] += 5.0

pca_train = PCA(n_components=2).fit(X_tr)
pca_leaked = PCA(n_components=2).fit(np.vstack([X_tr, X_te]))

np.testing.assert_allclose(pca_train.mean_, X_tr.mean(axis=0), atol=1e-12)
assert abs(pca_train.mean_[0] - pca_leaked.mean_[0]) > 1.5
print("Média feature 0 — fit no treino:", round(float(pca_train.mean_[0]), 6))
print("Média feature 0 — fit treino+teste:", round(float(pca_leaked.mean_[0]), 6))

## 5. t-SNE: vizinhanças, perplexidade e seed

Usaremos `load_digits`, distribuído com scikit-learn. Cada execução recebe exatamente os mesmos dados; mudam apenas perplexidade e seed. `init="random"` torna a sensibilidade observável.

In [ ]:
digits = load_digits()
idx = rng.choice(len(digits.data), size=700, replace=False)
X_digits = digits.data[idx] / 16.0
y_digits = digits.target[idx]

# PCA inicial reduz ruído e custo sem usar rótulos.
X_digits_30 = PCA(n_components=30, random_state=SEED).fit_transform(X_digits)
print("Shape original:", X_digits.shape)
print("Shape para os mapas:", X_digits_30.shape)

In [ ]:
def tsne_map(perplexity, seed):
    return TSNE(
        n_components=2,
        perplexity=perplexity,
        init="random",
        learning_rate="auto",
        max_iter=500,
        random_state=seed,
    ).fit_transform(X_digits_30)

map_p5 = tsne_map(5, SEED)
map_p40_a = tsne_map(40, SEED)
map_p40_b = tsne_map(40, SEED + 1)

tw = {
    "p5": trustworthiness(X_digits_30, map_p5, n_neighbors=10),
    "p40_seed_a": trustworthiness(X_digits_30, map_p40_a, n_neighbors=10),
    "p40_seed_b": trustworthiness(X_digits_30, map_p40_b, n_neighbors=10),
}
assert min(tw.values()) > 0.93
print("Trustworthiness@10:", {k: round(float(v), 6) for k, v in tw.items()})

### O que permanece e o que muda

Trustworthiness alta indica poucos vizinhos falsos na escala escolhida. Ela não prova classes verdadeiras, preservação global ou estabilidade. Para mostrar isso, comparamos distâncias entre os mesmos pares de observações.

In [ ]:
pair_rng = np.random.default_rng(SEED + 99)
pairs = pair_rng.integers(0, len(X_digits_30), size=(2500, 2))
pairs = pairs[pairs[:, 0] != pairs[:, 1]]

def paired_distances(A):
    return np.linalg.norm(A[pairs[:, 0]] - A[pairs[:, 1]], axis=1)

d_high = paired_distances(X_digits_30)
d_p5 = paired_distances(map_p5)
d_p40_a = paired_distances(map_p40_a)
d_p40_b = paired_distances(map_p40_b)

rho_global = spearmanr(d_high, d_p40_a).statistic
rho_perplexity = spearmanr(d_p5, d_p40_a).statistic
rho_seed = spearmanr(d_p40_a, d_p40_b).statistic

assert rho_global < 0.75
assert rho_perplexity < 0.85
assert rho_seed < 0.999
print(f"Spearman distâncias: original×t-SNE={rho_global:.6f}")
print(f"t-SNE perplexidade 5×40={rho_perplexity:.6f}")
print(f"t-SNE seed A×B={rho_seed:.6f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
for ax, embedding, title in zip(
    axes,
    [map_p5, map_p40_a, map_p40_b],
    ["perplexity=5", "perplexity=40", "perplexity=40, outra seed"],
):
    scatter = ax.scatter(embedding[:, 0], embedding[:, 1], c=y_digits, s=8, cmap="tab10")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("t-SNE: os rótulos são usados apenas para colorir depois do ajuste")
plt.close(fig)
print("Figura validada em memória:", len(axes), "painéis")

## 6. Antes do layout UMAP: grafo fuzzy de vizinhos

O UMAP completo constrói um grafo fuzzy e otimiza um layout de baixa dimensão. Aqui implementamos o primeiro estágio: distância local $\rho_i$, escala $\sigma_i$, pertenças direcionadas e união fuzzy. Não chamaremos esse grafo de “mapa UMAP”.

In [ ]:
def smooth_knn_sigma(distances, rho, target, n_iter=64):
    lo, hi = 1e-6, 1e3
    for _ in range(n_iter):
        mid = (lo + hi) / 2
        value = np.exp(-np.maximum(0.0, distances - rho) / mid).sum()
        if value > target:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2

def fuzzy_knn_graph(X_input, n_neighbors):
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(X_input)
    distances, indices = nn.kneighbors(X_input)
    distances, indices = distances[:, 1:], indices[:, 1:]
    rhos = distances[:, 0]
    target = np.log2(n_neighbors)
    sigmas = np.array([
        smooth_knn_sigma(row, rho, target)
        for row, rho in zip(distances, rhos)
    ])
    directed_weights = np.exp(
        -np.maximum(0.0, distances - rhos[:, None]) / sigmas[:, None]
    )
    rows = np.repeat(np.arange(len(X_input)), n_neighbors)
    directed = sparse.csr_matrix(
        (directed_weights.ravel(), (rows, indices.ravel())),
        shape=(len(X_input), len(X_input)),
    )
    # União fuzzy: a + b - ab.
    union = directed + directed.T - directed.multiply(directed.T)
    union.eliminate_zeros()
    return union.tocsr(), rhos, sigmas

X_graph = X_digits_30[:500]
graph5, rho5, sigma5 = fuzzy_knn_graph(X_graph, 5)
graph30, rho30, sigma30 = fuzzy_knn_graph(X_graph, 30)

def graph_summary(graph):
    components = connected_components(graph, directed=False, return_labels=False)
    density = graph.nnz / (graph.shape[0] * (graph.shape[0] - 1))
    return components, density

components5, density5 = graph_summary(graph5)
components30, density30 = graph_summary(graph30)

np.testing.assert_allclose(graph5.toarray(), graph5.T.toarray(), atol=1e-12)
assert graph5.data.min() >= 0 and graph5.data.max() <= 1
assert density30 > density5
assert components30 <= components5
print(f"k=5: componentes={components5}; densidade={density5:.6f}")
print(f"k=30: componentes={components30}; densidade={density30:.6f}")

### Escalas locais

$\rho_i$ conecta o vizinho mais próximo com pertença máxima; $\sigma_i$ adapta a queda das demais arestas à densidade local. Distribuições diferentes dessas escalas mostram que o grafo não usa um único raio global.

In [ ]:
assert np.all(rho5 >= 0)
assert np.all(sigma5 > 0)
assert np.std(sigma5) > 0

fig, axes = plt.subplots(1, 2, figsize=(8, 3), constrained_layout=True)
axes[0].hist(rho5, bins=20)
axes[0].set(title="Distância local rho (k=5)", xlabel="rho", ylabel="contagem")
axes[1].hist(sigma5, bins=20)
axes[1].set(title="Escala local sigma (k=5)", xlabel="sigma", ylabel="contagem")
plt.close(fig)
print(f"rho: mediana={np.median(rho5):.6f}")
print(f"sigma: mediana={np.median(sigma5):.6f}; desvio={np.std(sigma5):.6f}")

## 7. Protocolo reutilizável

1. Defina se o objetivo é compressão, modelagem ou visualização.
2. Faça o split antes de qualquer `fit` quando houver avaliação supervisionada.
3. Coloque PCA e escala dentro do pipeline.
4. Meça a tarefa downstream; variância explicada é diagnóstico, não objetivo universal.
5. Para mapas, repita seeds e escalas de vizinhança.
6. Registre versão, seed, amostra, métrica, pré-processamento e limites da leitura.

## Extensão opcional: layout UMAP completo

Em ambiente com `umap-learn`, instale uma versão compatível e execute `UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(X_digits_30)`. Compare ao menos três valores de `n_neighbors`, três seeds e `trustworthiness`. A dependência é opcional porque a API e as versões compatíveis podem mudar; registre a versão instalada.

## Verificações automáticas executadas

- equivalência entre PCA manual e scikit-learn;
- erro de reconstrução não crescente;
- sensibilidade do PCA às unidades;
- contraexemplo de sinal preditivo em baixa variância;
- média do PCA aprendida apenas no treino;
- trustworthiness dos mapas t-SNE;
- distorção/instabilidade de distâncias globais;
- simetria, limites e densidade do grafo fuzzy.

## Conclusão

PCA fornece uma transformação linear reproduzível e aplicável a novos dados, desde que o ajuste respeite o split. t-SNE oferece um mapa de vizinhanças, não uma régua global. O UMAP começa por um grafo de afinidades locais e só depois otimiza o layout. Em todos os casos, a projeção precisa ser avaliada segundo o uso declarado.

Na próxima aula, esse raciocínio será ampliado para provenance, reprodutibilidade e pipelines sem vazamento.